# Haar Cascade preprocessing

Place the raw videos in `./real_yt/` and `./fake_yt/` (relative to the project root) before running. Face-cropped 112×112 videos are written to `./real_yt_face_videos/` and `./fake_yt_face_videos/`.


In [ ]:
#Frame-count statistics for the raw real videos
import glob
import numpy as np
import cv2

video_files = []
frame_count = []
for video_file in glob.glob('./real_yt/*.mp4'):
    cap = cv2.VideoCapture(video_file)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if n < 150:
        continue
    video_files.append(video_file)
    frame_count.append(n)
print("frames:", frame_count)
print("Total number of videos:", len(frame_count))
print("Average frames per video:", np.mean(frame_count))


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
# You can also check how many GPUs and their names:
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce MX450


In [3]:
import os
import glob
import cv2
import numpy as np
from tqdm.auto import tqdm

def frame_extract(path, max_frames=150):
    """Yield up to max_frames BGR frames (as uint8 arrays)."""
    vid = cv2.VideoCapture(path)
    count = 0
    while count < max_frames:
        success, frame = vid.read()
        if not success or frame is None:
            break
        yield frame
        count += 1
    vid.release()

def create_face_videos(video_paths, out_dir):
    """Use OpenCV Haar cascade to crop faces and save 112×112 MP4s."""
    os.makedirs(out_dir, exist_ok=True)
    cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    
    for src in tqdm(video_paths, desc="Videos"):
        dst = os.path.join(out_dir, os.path.basename(src))
        if os.path.exists(dst):
            continue
        
        writer = cv2.VideoWriter(dst, fourcc, 30, (112,112), True)
        frames_written = 0
        
        for frame in frame_extract(src):
            # detect faces in grayscale
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = cascade.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(50,50)
            )
            if len(faces) == 0:
                continue
            
            # take the first face
            x,y,w,h = faces[0]
            face = frame[y:y+h, x:x+w]
            # resize & write
            face = cv2.resize(face, (112,112))
            writer.write(face)
            frames_written += 1
            
            # stop once we have 150 frames
            if frames_written >= 150:
                break
        
        writer.release()
        print(f"Saved: {dst} ({frames_written} frames)")


In [4]:
video_files = glob.glob('./real_yt/*.mp4')  # Adjust path as needed
create_face_videos(video_files, './real_yt_face_videos/')

Videos:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: ./real_yt_face_videos/030.mp4 (150 frames)
Saved: ./real_yt_face_videos/241.mp4 (150 frames)


In [5]:
video_files = glob.glob('./fake_yt/*.mp4')
create_face_videos(video_files, './fake_yt_face_videos/')

Videos:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: ./fake_yt_face_videos/014_790.mp4 (150 frames)
Saved: ./fake_yt_face_videos/286_267.mp4 (150 frames)
